<a href="https://colab.research.google.com/github/hwangho-kim/Transformer_Fewshot_PdM/blob/main/RUL_%EC%98%88%EC%B8%A1_%EB%B0%8F_%EC%8B%9C%EA%B0%81%ED%99%94_%EC%BD%94%EB%93%9C_VTR_R03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Windows, Mac, Linux 환경에 맞게 자동 설정)
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin': # Mac
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지


def generate_dummy_data(file_path):
    """테스트를 위한 가상 시계열 센서 데이터 생성 함수"""
    print("가상 데이터를 생성합니다...")
    np.random.seed(42)

    dates = pd.date_range(start='2023-01-01', periods=10000, freq='10min')
    mean_values = []

    current_val = 100
    for i in range(10000):
        # 약간의 노이즈와 함께 점진적으로 상승 (열화 현상 모사)
        current_val += np.random.normal(1.5, 5.0)
        if current_val >= 1100:
            mean_values.append(1100 + np.random.normal(0, 2)) # 고장 시점
            current_val = 100 # 부품 교체 후 초기화
        else:
            mean_values.append(current_val)

    df = pd.DataFrame({'start_time': dates, 'mean': mean_values})
    df.to_parquet(file_path)
    print(f"가상 데이터 저장 완료: {file_path}")


def prepare_rul_data(parquet_path):
    """Parquet 데이터 로드 및 RUL 타겟/피처 엔지니어링"""
    print("\n1. 파케이(Parquet) 데이터 로드 중...")
    df = pd.read_parquet(parquet_path)

    if 'start_time' not in df.columns or 'mean' not in df.columns:
        raise ValueError("데이터에 'start_time'과 'mean' 컬럼이 필요합니다.")

    df['start_time'] = pd.to_datetime(df['start_time'])
    df = df.sort_values('start_time').reset_index(drop=True)

    print("2. RUL(잔여 수명) 레이블링 진행 중...")
    # 임계치 1100 도달 시점을 고장으로 판단
    df['is_failure'] = (df['mean'] >= 1100).astype(int)

    # 새로운 부품 교체 주기(Cycle) 식별
    df['cycle_id'] = df['is_failure'].shift().fillna(0).cumsum()

    # 각 사이클의 고장 시간 계산 (기존 end_time 컬럼과의 충돌 방지를 위해 cycle_end_time으로 명명)
    cycle_end_times = df.groupby('cycle_id')['start_time'].max().rename('cycle_end_time')
    df = df.join(cycle_end_times, on='cycle_id')

    # RUL 계산 (현재 시점에서 고장 시간까지 남은 시간을 '분' 단위로 계산)
    df['RUL'] = (df['cycle_end_time'] - df['start_time']).dt.total_seconds() / 60.0

    print("3. 시계열 피처 엔지니어링 진행 중...")
    # 이동 평균, 표준 편차, 변화량 등 시계열 패턴 피처 생성
    df['mean_roll_mean_5'] = df.groupby('cycle_id')['mean'].rolling(window=5, min_periods=1).mean().reset_index(0, drop=True)
    df['mean_roll_std_5'] = df.groupby('cycle_id')['mean'].rolling(window=5, min_periods=1).std().reset_index(0, drop=True).fillna(0)
    df['mean_diff_1'] = df.groupby('cycle_id')['mean'].diff(1).fillna(0)
    df['hour'] = df['start_time'].dt.hour

    # 불필요한 컬럼은 보관을 위해 분리하고 학습용 데이터셋 구성 (기존 end_time도 함께 무시되도록 처리)
    meta_cols = ['start_time', 'cycle_end_time', 'is_failure', 'cycle_id', 'mean']
    X = df.drop(columns=['cycle_end_time', 'end_time', 'is_failure', 'RUL'] + (['start_time', 'cycle_id', 'mean'] if False else []), errors='ignore')
    y = df['RUL']

    return df, X, y


def train_and_visualize(df_raw, X, y):
    """XGBoost 모델 학습 및 시각화"""
    print("4. 데이터 분할 (시간 순 8:2 분할)...")

    # 메타 정보 보존 (차트 그릴 때 사용하기 위함)
    features = X.drop(columns=['start_time', 'cycle_id'])

    split_idx = int(len(features) * 0.8)
    X_train, X_test = features.iloc[:split_idx], features.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    df_test = df_raw.iloc[split_idx:].reset_index(drop=True)

    print("5. XGBoost GPU 모델 학습 시작...")
    # P100 GPU 가속 활성화 (device='cuda')
    model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        tree_method='hist',
        device='cuda', # GPU 가속
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        early_stopping_rounds=50 # 최신 버전 규격에 맞게 위치 이동
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_test, y_test)],
        verbose=100
    )

    print("\n6. 모델 예측 및 평가...")
    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    print(f"Test RMSE: {rmse:.2f} 분")
    print(f"Test MAE: {mae:.2f} 분")

    # 시각화 (테스트 셋 중 특정 사이클 1개만 추출하여 자세히 보기)
    plot_cycle_id = df_test[df_test['is_failure'] == 1]['cycle_id'].iloc[0]
    cycle_mask = df_test['cycle_id'] == plot_cycle_id

    plot_time = df_test.loc[cycle_mask, 'start_time']
    plot_mean = df_test.loc[cycle_mask, 'mean']
    plot_actual_rul = df_test.loc[cycle_mask, 'RUL']
    plot_pred_rul = preds[cycle_mask]

    print("\n7. 차트 생성 중...")
    plt.figure(figsize=(14, 10))

    # [차트 1] 센서 데이터 열화 추이
    plt.subplot(2, 1, 1)
    plt.plot(plot_time, plot_mean, color='#1f77b4', linewidth=2, label='Sensor Data (mean)')
    plt.axhline(y=1100, color='red', linestyle='--', linewidth=2, label='Failure Threshold (1100)')
    plt.title('Semiconductor Equipment Sensor Data Degradation Trend', fontsize=16, fontweight='bold')
    plt.ylabel('Sensor Value (mean)', fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.legend(fontsize=12)

    # [차트 2] 잔여 수명(RUL) 실제값 vs 예측값
    plt.subplot(2, 1, 2)
    plt.plot(plot_time, plot_actual_rul, color='black', linestyle='-', linewidth=2, label='Actual RUL')
    plt.plot(plot_time, plot_pred_rul, color='#ff7f0e', linestyle='--', linewidth=2.5, alpha=0.8, label='Predicted RUL')

    # RUL이 0이 되는 지점(고장 예측 시점) 강조
    plt.axhline(y=0, color='gray', linestyle='-')
    plt.title('Equipment Remaining Useful Life (RUL) Prediction Monitoring', fontsize=16, fontweight='bold')
    plt.xlabel('Time', fontsize=12)
    plt.ylabel('Remaining Useful Life (Minutes)', fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.legend(fontsize=12)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    parquet_file = 'sample_sensor_data.parquet'

    # 1. 파일이 없으면 가상의 샘플 데이터 생성
    if not os.path.exists(parquet_file):
        generate_dummy_data(parquet_file)

    try:
        # 2. 데이터 준비
        df_raw, X, y = prepare_rul_data(parquet_file)

        # 3. 학습 및 시각화 차트 출력
        train_and_visualize(df_raw, X, y)

    except Exception as e:
        print(f"오류가 발생했습니다: {e}")